<a href="https://colab.research.google.com/github/anoushka-pandey/CLNLP_LAB/blob/main/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Experiment-5: Subword Level Tokenization and POS Tagging**

In [13]:
file_path = "/content/drive/MyDrive/CLNLP_Lab/input_sub_word_data.txt"

### **5.1. Implement subword-level tokenization using a BPE-based tokenizer.**
**a. Using Pretrained Model**
1. Tokenize the sentence into subword units.
2. Display the generated tokens and their corresponding token IDs.




In [14]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
with open(file_path, "r", encoding="utf-8") as f:
    sample_text = f.readline().strip()

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("=== 5.1 (a) BPE Pretrained ===")
print("Original Text:", sample_text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

=== 5.1 (a) BPE Pretrained ===
Original Text: Natural language processing is a branch of artificial intelligence that
Tokens: ['Natural', 'Ġlanguage', 'Ġprocessing', 'Ġis', 'Ġa', 'Ġbranch', 'Ġof', 'Ġartificial', 'Ġintelligence', 'Ġthat']
Token IDs: [35364, 3303, 7587, 318, 257, 8478, 286, 11666, 4430, 326]


**b. Without using Pretrained Model**
1. Tokenize the sentence into subword units.
2. Display the generated tokens and their corresponding token IDs.


In [15]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Initialize empty BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. Train it on your specific file (Vocabulary size 500 for demonstration)
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=500)
tokenizer.train(files=[file_path], trainer=trainer)

# 3. Test on a sentence
with open(file_path, "r", encoding="utf-8") as f:
    sample_text = f.readline().strip()

output = tokenizer.encode(sample_text)

print("=== 5.1 (b) BPE Trained from Scratch ===")
print("Tokens:", output.tokens)
print("Token IDs:", output.ids)

=== 5.1 (b) BPE Trained from Scratch ===
Tokens: ['Natural', 'language', 'processing', 'is', 'a', 'b', 'r', 'an', 'ch', 'of', 'ar', 't', 'ific', 'ial', 'in', 't', 'el', 'l', 'ig', 'ence', 'that']
Token IDs: [356, 139, 189, 79, 21, 22, 38, 51, 81, 86, 56, 40, 441, 377, 48, 40, 102, 32, 304, 175, 263]


### **5.2. Implement subword-level tokenization using a SentencePiece tokenizer.**
**a. Using a Pretrained Model**
1. Tokenize the sentence into subword units.
2. Display the generated tokens and their corresponding token IDs.


In [16]:
from transformers import T5Tokenizer

# 1. Load pretrained SentencePiece tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

with open(file_path, "r", encoding="utf-8") as f:
    sample_text = f.readline().strip()

# 2. Tokenize and get IDs (SentencePiece uses '_' to denote spaces)
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("=== 5.2 (a) SentencePiece Pretrained ===")
print("Tokens:", tokens)
print("Token IDs:", token_ids)

=== 5.2 (a) SentencePiece Pretrained ===
Tokens: ['▁Natural', '▁language', '▁processing', '▁is', '▁', 'a', '▁branch', '▁of', '▁artificial', '▁intelligence', '▁that']
Token IDs: [6869, 1612, 3026, 19, 3, 9, 6421, 13, 7353, 6123, 24]


**b. Using a Pretrained Model**
1. Tokenize the sentence into subword units.
2. Display the generated tokens and their corresponding token IDs.

In [20]:
import sentencepiece as spm
import shutil
import os

# Create a local temporary path
local_file_path = "/tmp/input_sub_word_data.txt"

# Copy the file from Google Drive to the local filesystem
shutil.copy(file_path, local_file_path)

# 1. Train SentencePiece model on your file (Creates 'm.model' and 'm.vocab')
spm.SentencePieceTrainer.train(input=local_file_path, model_prefix='custom_spm', vocab_size=300)

# 2. Load the trained model
sp = spm.SentencePieceProcessor()
sp.load('custom_spm.model')

with open(file_path, "r", encoding="utf-8") as f:
    sample_text = f.readline().strip()

# 3. Tokenize and get IDs
tokens = sp.encode_as_pieces(sample_text)
token_ids = sp.encode_as_ids(sample_text)

print("=== 5.2 (b) SentencePiece Trained from Scratch ===")
print("Tokens:", tokens)
print("Token IDs:", token_ids)

# Optional: Clean up the local temporary file
os.remove(local_file_path)

=== 5.2 (b) SentencePiece Trained from Scratch ===
Tokens: ['▁Natural', '▁language', '▁processing', '▁is', '▁a', '▁', 'b', 'r', 'a', 'nc', 'h', '▁of', '▁arti', 'fic', 'i', 'al', '▁in', 'te', 'l', 'l', 'i', 'g', 'ence', '▁that']
Token IDs: [185, 28, 54, 22, 8, 3, 71, 25, 27, 277, 299, 13, 250, 91, 12, 47, 40, 57, 16, 16, 12, 298, 144, 104]


### **5.3. Implement Part-of-Speech (POS) tagging on a given text and display the unique token, POS tag, and description of each POS tag.**

**a. Using SpaCy**


In [10]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "The young student is reading an interesting book in the library."

doc = nlp(text)

print("=== 5.3 (a) POS Tagging using SpaCy ===")
print(f"{'Token':<15} | {'POS Tag':<10} | {'Description'}")
print("-" * 50)
for token in doc:
    print(f"{token.text:<15} | {token.pos_:<10} | {spacy.explain(token.pos_)}")

=== 5.3 (a) POS Tagging using SpaCy ===
Token           | POS Tag    | Description
--------------------------------------------------
The             | DET        | determiner
young           | ADJ        | adjective
student         | NOUN       | noun
is              | AUX        | auxiliary
reading         | VERB       | verb
an              | DET        | determiner
interesting     | ADJ        | adjective
book            | NOUN       | noun
in              | ADP        | adposition
the             | DET        | determiner
library         | NOUN       | noun
.               | PUNCT      | punctuation


**b. Using NLTK**

In [22]:
import nltk
from nltk.tokenize import word_tokenize

# Download required NLTK packages
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('tagsets')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # Added to download the specific English tagger

text = "The young student is reading an interesting book in the library."

# Tokenize and Tag
tokens = word_tokenize(text)
pos_tags = nltk.pos_tag(tokens)

# Load tag dictionary for descriptions
tagdict = nltk.data.load('help/tagsets/upenn_tagset.pickle')

print("=== 5.3 (b) POS Tagging using NLTK ===")
print(f"{'Token':<15} | {'POS Tag':<10} | {'Description'}")
print("-" * 55)
for token, tag in pos_tags:
    # Fetch description from tagdict
    description = tagdict.get(tag, ["Unknown"])[0]
    print(f"{token:<15} | {tag:<10} | {description}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /root/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


=== 5.3 (b) POS Tagging using NLTK ===
Token           | POS Tag    | Description
-------------------------------------------------------
The             | DT         | determiner
young           | JJ         | adjective or numeral, ordinal
student         | NN         | noun, common, singular or mass
is              | VBZ        | verb, present tense, 3rd person singular
reading         | VBG        | verb, present participle or gerund
an              | DT         | determiner
interesting     | JJ         | adjective or numeral, ordinal
book            | NN         | noun, common, singular or mass
in              | IN         | preposition or conjunction, subordinating
the             | DT         | determiner
library         | NN         | noun, common, singular or mass
.               | .          | sentence terminator


### **5.4. Implement Part-of-Speech (POS) tagging on a given text and display the unique token, POS tag, description of each token and frequency using Spacy.**

In [19]:
import spacy
from collections import Counter

# 1. Load SpaCy model
nlp = spacy.load("en_core_web_sm")

# 2. Read the entire file
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# 3. Process text using SpaCy
doc = nlp(text)

# 4. Extract unique tokens, POS, description and count frequencies
# We ignore spaces/newlines to keep the output clean
pos_data = []
for token in doc:
    if not token.is_space:
        # Convert to lowercase to group same words together (e.g., "The" and "the")
        pos_data.append((token.text.lower(), token.pos_, spacy.explain(token.pos_)))

# Count occurrences
frequency_counts = Counter(pos_data)

# 5. Display the results
print("=== 5.4 POS Tagging with Frequencies on File ===")
print(f"{'Token':<18} | {'POS Tag':<10} | {'Description':<25} | {'Frequency'}")
print("-" * 70)

# Displaying only the top 20 most frequent for a clean output
for (word, pos, desc), freq in frequency_counts.most_common(20):
    print(f"{word:<18} | {pos:<10} | {str(desc):<25} | {freq}")

=== 5.4 POS Tagging with Frequencies on File ===
Token              | POS Tag    | Description               | Frequency
----------------------------------------------------------------------
,                  | PUNCT      | punctuation               | 76
.                  | PUNCT      | punctuation               | 65
the                | DET        | determiner                | 47
a                  | DET        | determiner                | 28
and                | CCONJ      | coordinating conjunction  | 27
can                | AUX        | auxiliary                 | 23
of                 | ADP        | adposition                | 22
be                 | AUX        | auxiliary                 | 15
is                 | AUX        | auxiliary                 | 14
as                 | ADP        | adposition                | 13
bpe                | PROPN      | proper noun               | 13
language           | NOUN       | noun                      | 12
word               | NOUN   